# Data Extraction

Author: **Quang-Huy Tran**, Student at HCMC University of Technology, VNU-HCM  
Dataset: ICDAR2003. https://bibtex.github.io/ICDAR-2003.html  
Model: YOLOv11

In [4]:
!unzip ICDAR2003.zip

Archive:  ICDAR2003.zip
   creating: data/
   creating: data/SceneTrialTrain/
   creating: data/SceneTrialTrain/apanar_06.08.2002/
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1247.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1252.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1253.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1255.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1259.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1261.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1263.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1265.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1269.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1281.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1282.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1283.JPG  
  inflating: data/SceneTrialTrain/apanar_06.08.2002/IMG_1284.JPG 

In [27]:
import xml.etree.ElementTree as ET
import os

### Đọc file XML.
XML là một Tree khổng lồ.  
Ta cần đọc file `words.xml`. Sau khi đọc, `root` chứa toàn bộ thông tin về 4 folder ảnh (là các `<tagset>`)  
Một thẻ trong XML tồn tại dưới dạng object. Nếu nó có node con, ta có thể truy cập thông qua vòng lặp. Ở node con, truy cập tới `node.tag` để lấy tên thẻ, hoặc `<node.attrib>` để xem dictionary chứa các attribute.

In [6]:
pathfile = './data/SceneTrialTrain/words.xml'
tree = ET.parse(pathfile)
root = tree.getroot()
root

<Element 'tagset' at 0x7c0a69204540>

Môi (`<tagset>`) chứa các `<image>`, tức thông tin của các ảnh.

In [7]:
# 250 ảnh, mỗi ảnh có 3 thông tin (imageName, resolution, taggedRectangles)
len(root), len(root[0])

(250, 3)

Ảnh đầu tiên là `root[0]`. các phần tử của ảnh đầu tiên bao gồm:
- `root[0][0]`: imageName, dùng `text` để lấy ra.
- `root[0][1]`: độ phân giải (resolution), dùng `attrib['x']` hoặc `y` để lấy ra.

In [8]:
root[0][0].text

'apanar_06.08.2002/IMG_1261.JPG'

In [9]:
root[0][1].attrib['x'], root[0][1].attrib['y']

('1600', '1200')

- `root[0][2]`: **taggedRectangle**, tức các bounding box của ảnh đầu tiên - bao nhiêu bbox thì bấy nhiêu object được phát hiện. Như vậy chính nó chứa nhiều **taggedRectangle** khác.  
- tồn tại dưới dạng object `<Element 'taggedRectangles' at 0x7c0b6ce79da0>`. Ta có thể truy cập vào từng bounding_box sau đó gọi `attrib` sẽ lấy được các số liệu của 1 bounding box trong ảnh.

In [10]:
# tất cả bounding box của ảnh 0
root[0][2]

<Element 'taggedRectangles' at 0x7c0a692051c0>

In [11]:
# bounding box thứ 1 của ảnh 0
root[0][2][1]

<Element 'taggedRectangle' at 0x7c0a69205300>

In [12]:
# bounding box thứ 1 của ảnh 0 có 2 con, bao gồm chữ mà nó chứa + segmentation.
# con đầu tiên chính là chữ mà nó chứa.
root[0][2][1][0].text

'adhesive'

In [13]:
# bounding box thứ 1 của ảnh 0 chứa các thuộc tính:
root[0][2][1].attrib

{'x': '512.0',
 'y': '391.0',
 'width': '679.0',
 'height': '183.0',
 'offset': '0.0',
 'rotation': '0.0',
 'userName': 'admin'}

In [14]:
# lấy mọi bounding box trong ảnh số 0 và in ra.
for bb in root[0][2]:
  bbox = [
  bb.attrib['x'],
  bb.attrib['y'],
  bb.attrib['width'],
  bb.attrib['height']
  ]
  print(bbox)


['174.0', '392.0', '274.0', '195.0']
['512.0', '391.0', '679.0', '183.0']
['184.0', '612.0', '622.0', '174.0']
['863.0', '599.0', '446.0', '187.0']
['72.0', '6.0', '95.0', '87.0']
['247.0', '2.0', '197.0', '88.0']
['792.0', '0.0', '115.0', '81.0']
['200.0', '848.0', '228.0', '139.0']
['473.0', '878.0', '165.0', '109.0']
['684.0', '878.0', '71.0', '106.0']
['806.0', '844.0', '218.0', '141.0']


In [15]:
# ảnh 0: lấy mọi bounding box, in ra chữ + tọa độ.
image_0 = root[0][2]
for bb in image_0:
  object_name = bb[0].text
  bb_axis = [
      bb.attrib['x'],
      bb.attrib['y'],
      bb.attrib['width'],
      bb.attrib['height']
  ]
  print(object_name, '--', bb_axis)


self -- ['174.0', '392.0', '274.0', '195.0']
adhesive -- ['512.0', '391.0', '679.0', '183.0']
address -- ['184.0', '612.0', '622.0', '174.0']
labels -- ['863.0', '599.0', '446.0', '187.0']
36 -- ['72.0', '6.0', '95.0', '87.0']
89m -- ['247.0', '2.0', '197.0', '88.0']
cls -- ['792.0', '0.0', '115.0', '81.0']
250 -- ['200.0', '848.0', '228.0', '139.0']
on -- ['473.0', '878.0', '165.0', '109.0']
a -- ['684.0', '878.0', '71.0', '106.0']
roll -- ['806.0', '844.0', '218.0', '141.0']


 # hiện thực trong project
 - Viết hàm cơ bản: nhận đầu vào là `path` là đường dẫn tới root, đầu ra là:  
    + bounding box dạng là list ảnh, mỗi phần tử trong list là một list các bounding box. Dạng: `List[list[list[float]]]`, mỗi bounding box gồm `(x, y, height, weight)`. Độ dài của danh sách này bằng số ảnh trong data.
    + label dạng text, là object của bounding box (ánh xạ 1:1 giữa bounding box và label)
    + size dạng `tuple(a, b)`, chứa độ phân giải (resolution) của ảnh. Đầu ra là list các tuple, độ dài bằng tổng số ảnh.
    + `image_path`, là đường dẫn tới các ảnh.

In [57]:
def extract_data_from_xml(path):
  bbox_list = []
  label_list = []
  size_list = []
  img_path_list = []

  # XML library
  tree = ET.parse(path + 'words.xml')
  root = tree.getroot()

  for image in root:
    bbox_image = []
    label_image = []

    # get 'taggedRectangle' for label_list and bbox_list
    for bb_infor in image[2]:

      # UPDATE: check non-number and non-alphabet
      if not bb_infor[0].text.isalnum():
        continue

      # UPDATE: fix data error
      if "é" in bb_infor[0].text.lower() or "ñ" in bb_infor[0].text.lower():
        continue

      # get bbox and label
      bb = [
          float(bb_infor.attrib['x']),
          float(bb_infor.attrib['y']),
          float(bb_infor.attrib['width']),
          float(bb_infor.attrib['height']),
      ]
      bb_label = bb_infor[0].text.lower()


      bbox_image.append(bb)
      label_image.append(bb_label)

    # get path and size
    img_name = image[0].text
    img_path = os.path.join(path, img_name)
    img_size = (int(image[1].attrib['x']), int(image[1].attrib['y']))

    # store
    img_path_list.append(img_path)
    size_list.append(img_size)
    bbox_list.append(bbox_image)
    label_list.append(label_image)

  return img_path_list, size_list, bbox_list, label_list

In [60]:
PATH = './data/SceneTrialTrain/'
print(f'Image path example: {extract_data_from_xml(PATH)[0][:2]}')
print(f'Image resolution example: {extract_data_from_xml(PATH)[1][:2]}')
print(f'Bounding boxes example: {extract_data_from_xml(PATH)[2][0][:2]}')
print(f'Image resolution example: {extract_data_from_xml(PATH)[3][0][:2]}')

Image path example: ['./data/SceneTrialTrain/apanar_06.08.2002/IMG_1261.JPG', './data/SceneTrialTrain/apanar_06.08.2002/IMG_1263.JPG']
Image resolution example: [(1600, 1200), (1600, 1200)]
Bounding boxes example: [[174.0, 392.0, 274.0, 195.0], [512.0, 391.0, 679.0, 183.0]]
Image resolution example: ['self', 'adhesive']
